In [ ]:
from google.colab import files
files.upload()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
email = pd.read_csv("Phishing_Email.csv")

email.head()

,Unnamed: 0,Email Text,Email Type
0,0,"re : 6 . 1100 , disc : uniformitarianism , re ...",Safe Email
1,1,the other side of * galicismos * * galicismo *...,Safe Email
2,2,re : equistar deal tickets are you still avail...,Safe Email
3,3,\nHello I am your hot lil horny toy.\n I am...,Phishing Email
4,4,software at incredibly low prices ( 86 % lower...,Phishing Email


In [ ]:
print(email.columns)
print(email.isnull().sum())

Index(['Unnamed: 0', 'Email Text', 'Email Type'], dtype='object')
Unnamed: 0     0
Email Text    16
Email Type     0
dtype: int64


# Data preprocessing

In [ ]:
import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+", "URL", text)  # replace links
    text = re.sub(r"\W", " ", text)
    return text

email['clean_text'] = email['Email Text'].apply(clean_text)

In [ ]:
df = email.dropna()

# Feature Engineering 1


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=5000)
X = vectorizer.fit_transform(df['clean_text'])
y = df['Email Type']

In [ ]:
import os
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from datasets import Dataset
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)

# -----------------------
# CHECK GPU
# -----------------------
print("GPU Available:", torch.cuda.is_available())

# -----------------------
# LOAD DATA
# -----------------------
DATA_PATH = "Phishing_Email.csv"

LABEL_MAP = {
    "Phishing Email": 1,
    "Safe Email": 0
}

def parse_email_and_label(line):
    first_comma = line.find(",")
    if first_comma == -1:
        return None, None
    rest = line[first_comma + 1:]
    if not rest.startswith('"'):
        return None, None

    i = 1
    end_quote = None
    while i < len(rest):
        if rest[i] == '"':
            if i + 1 < len(rest) and rest[i + 1] == '"':
                i += 2
                continue
            end_quote = i
            break
        i += 1

    if end_quote is None:
        return None, None

    email_text = rest[1:end_quote].replace('""', '"')
    after = rest[end_quote + 1:]
    if not after.startswith(","):
        return None, None

    after = after[1:]
    next_comma = after.find(",")
    if next_comma == -1:
        label = after.strip()
    else:
        label = after[:next_comma].strip()

    if not label:
        return None, None
    return email_text, label


def load_records(path):
    texts, labels = [], []
    with open(path, "rb") as f:
        for raw in f:
            line = raw.decode("utf-8", errors="ignore").rstrip("\n\r")
            if not line:
                continue
            if line.startswith(",Email Text,Email Type"):
                continue
            text, label = parse_email_and_label(line)
            if label in LABEL_MAP:
                texts.append(text)
                labels.append(LABEL_MAP[label])
    return texts, labels


texts, labels = load_records(DATA_PATH)

# Optional: reduce dataset for faster experimentation
# texts = texts[:5000]
# labels = labels[:5000]

train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts, labels, test_size=0.2, random_state=42, stratify=labels
)

# -----------------------
# TOKENIZER
# -----------------------
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=128   # IMPORTANT SPEED BOOST
    )

train_dataset = Dataset.from_dict({"text": train_texts, "label": train_labels})
val_dataset = Dataset.from_dict({"text": val_texts, "label": val_labels})

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)

train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
val_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

# -----------------------
# MODEL
# -----------------------
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

# -----------------------
# METRICS
# -----------------------
def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "f1": f1, "precision": precision, "recall": recall}




In [ ]:
# -----------------------
# TRAINING
# -----------------------
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    load_best_model_at_end=True,
    fp16=True,  # GPU speed boost
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    tokenizer=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

trainer.train()

In [ ]:

# -----------------------
# SAVE MODEL
# -----------------------
model.save_pretrained("./phishing_model")
tokenizer.save_pretrained("./phishing_model")

print("✅ Training complete and model saved!")

In [ ]:
import shutil
shutil.make_archive("phishing_model", 'zip', "./phishing_model")

from google.colab import files
files.download("phishing_model.zip")

# Modeling

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = LogisticRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))

                precision    recall  f1-score   support

Phishing Email       0.96      0.96      0.96      1516
    Safe Email       0.97      0.97      0.97      2211

      accuracy                           0.96      3727
     macro avg       0.96      0.96      0.96      3727
  weighted avg       0.96      0.96      0.96      3727

